# 05 — Mini-YOLO from scratch

Classification: "what's in the image?" Detection: "**where** are all the things?" Different problem, different output, different loss.

## Roadmap
1. Detection problem framing
2. Bounding boxes and IoU (from scratch)
3. The grid-cell trick — YOLO's core idea
4. Non-Max Suppression in 20 lines
5. A toy YOLO on synthetic "find the bright square" data


In [ ]:
# === Bootstrap (safe to re-run) ===
# Installs minimal deps and downloads tiny CIFAR-10 subset for any notebook
# that needs it. The from-scratch notebooks deliberately avoid importing the
# project package so they run anywhere (Colab, plain Python, Jupyter).
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "matplotlib", "pillow", "torchvision"], check=True)
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("numpy", np.__version__)


## 1. The detection problem

Input: image. Output: a **variable-length** list of `(class, bounding_box, confidence)` tuples.

That variable-length-list output is the hard part. A CNN naturally produces a fixed-size tensor, not a list. YOLO's clever trick: **always predict a fixed-size grid, then post-process to a variable list.**

## 2. Bounding-box conventions

A box can be:

- **xyxy**: (x1, y1, x2, y2) corners
- **xywh**: (xc, yc, w, h) center + size
- **Pixel** or **normalised** to image size

Be ruthless about converting at the boundary of every function.


In [ ]:
import numpy as np

def xywh_to_xyxy(b):
    xc, yc, w, h = b
    return xc - w/2, yc - h/2, xc + w/2, yc + h/2

def iou(a, b):
    """IoU of two boxes in xyxy form."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    return inter / (area_a + area_b - inter + 1e-9)

# Sanity: identical boxes → 1.0, disjoint → 0.0
print(iou((0,0,4,4), (0,0,4,4)))                # 1.0
print(iou((0,0,4,4), (10,10,12,12)))            # 0.0
print(round(iou((0,0,4,4), (2,2,6,6)), 3))      # 0.143


## 3. The grid-cell idea

Divide the image into an $S \times S$ grid. **Each grid cell predicts:** `(p, x, y, w, h, c_0, c_1, ..., c_{N-1})`.

- `p`: objectness — is there anything here at all?
- `(x, y, w, h)`: the box (relative to the cell)
- `c_i`: per-class probability

For a 7×7 grid with 6 classes, the model output shape is `(7, 7, 1+4+6) = (7, 7, 11)`. That's a fixed tensor — a CNN can produce it.


In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(6, 6))
S = 7
for i in range(S+1):
    ax.axhline(i/S, color="gray", lw=0.5)
    ax.axvline(i/S, color="gray", lw=0.5)
# Pretend ground-truth box centered in cell (3, 2)
gt = (0.35, 0.28, 0.30, 0.25)  # xc, yc, w, h
x1, y1, x2, y2 = xywh_to_xyxy(gt)
ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="red", lw=2))
ax.plot(gt[0], gt[1], "ro"); ax.text(gt[0]+0.01, gt[1]-0.02, "object center", color="red")
ax.set_xlim(0,1); ax.set_ylim(1,0); ax.set_aspect("equal")
ax.set_title("Each cell predicts: 'is there an object whose center is in me?'")
plt.show()


## 4. Non-Max Suppression (NMS)

After the model predicts boxes everywhere, many of them overlap on the same object. NMS keeps the best one and suppresses the rest:

1. Sort boxes by confidence descending.
2. Pick the top box; add to keep list.
3. Discard remaining boxes with IoU > threshold.
4. Repeat.


In [ ]:
def nms(boxes, scores, iou_thresh=0.5):
    order = sorted(range(len(scores)), key=lambda i: -scores[i])
    keep = []
    while order:
        i = order.pop(0)
        keep.append(i)
        order = [j for j in order if iou(boxes[i], boxes[j]) < iou_thresh]
    return keep

# Three overlapping boxes around the same object, plus one separate one
boxes = [
    (0.10, 0.10, 0.40, 0.40),     # A — same object as B,C
    (0.12, 0.12, 0.42, 0.42),     # B
    (0.11, 0.09, 0.41, 0.43),     # C
    (0.60, 0.60, 0.90, 0.90),     # D — separate object
]
scores = [0.92, 0.85, 0.78, 0.95]
keep = nms(boxes, scores, iou_thresh=0.4)
print("kept boxes:", keep)        # → [3, 0]  (D first by score, then best of A/B/C)


## 5. Toy mini-YOLO on synthetic data

We'll generate 100×100 images, each with a single white square at a random location. The model learns to predict where the square is. Real YOLO is more complex (multi-scale features, focal loss, anchor-free distribution prediction), but this captures the *idea*.


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

def make_synthetic(n=400, size=64):
    imgs = np.zeros((n, 1, size, size), dtype=np.float32)
    labels = np.zeros((n, 4), dtype=np.float32)  # xc, yc, w, h (normalised)
    for k in range(n):
        w = np.random.randint(8, 20)
        h = np.random.randint(8, 20)
        x1 = np.random.randint(0, size - w)
        y1 = np.random.randint(0, size - h)
        imgs[k, 0, y1:y1+h, x1:x1+w] = 1.0
        labels[k] = [(x1 + w/2) / size, (y1 + h/2) / size, w / size, h / size]
    return imgs, labels

X_syn, y_syn = make_synthetic(400)
print("X:", X_syn.shape, " y:", y_syn.shape, " first label:", y_syn[0])

# Tiny CNN that regresses 4 numbers per image — a *simplified* detector
# (real YOLO has the grid + class outputs; we predict the single object's xywh).
class MiniDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(64, 4)

    def forward(self, x):
        return torch.sigmoid(self.head(self.body(x).flatten(1)))   # outputs in [0,1]

model = MiniDetector()
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
Xt = torch.tensor(X_syn); yt = torch.tensor(y_syn)

for ep in range(50):
    pred = model(Xt)
    loss = F.mse_loss(pred, yt)
    opt.zero_grad(); loss.backward(); opt.step()
    if (ep + 1) % 10 == 0:
        print(f"epoch {ep+1}  mse={loss.item():.4f}")

# Visualise a few predictions
import matplotlib.patches as patches
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
model.eval()
with torch.no_grad():
    preds = model(Xt[:4]).numpy()
for ax, img, gt, pr in zip(axes, X_syn[:4, 0], y_syn[:4], preds):
    ax.imshow(img, cmap="gray")
    for box, color in [(gt, "green"), (pr, "red")]:
        xc, yc, w, h = box * 64
        ax.add_patch(patches.Rectangle((xc-w/2, yc-h/2), w, h,
                                        fill=False, edgecolor=color, lw=2))
    ax.set_title("green=GT  red=pred"); ax.axis("off")
plt.show()


**Real YOLO** scales this idea with:
- Multi-scale feature maps (predict at 80×80, 40×40, 20×20 to catch big and small objects)
- Per-cell objectness + class probabilities
- Anchor-free distribution focal loss (YOLOv8)
- Heavy augmentation (MOSAIC, MixUp)

But the *core* — grid cells predicting box+class, with IoU + NMS at inference — is exactly what we just built.

**Next:** optimisation tricks (Adam, batch norm, dropout) that make any of these models actually trainable.
